# V09 – Dateien & Fehler: CSV einlesen und Fehler abfangen – Vertiefung zum Training

**Wo passt das Notebook hin?** Erst die Python-Werkstatt in der Lektion, dann das Training zu Hause (Browser, ca. 60 Min). Dieses Notebook ist die **Vertiefung**: Du übst dieselben Konzepte (Text zerlegen mit `strip`, `split`, `replace`, `float`; Dateien lesen und schreiben mit `with open`; das `csv`-Modul mit `delimiter` und `next`; `try`/`except` für kaputte Zeilen) an ingenieurnahen Aufgaben: Bremsmessung vom Fahrrad, Vibration der Waschmaschine.

**Die Beispieldateien erzeugt das Notebook selbst.** Die Zelle unten legt drei Dateien im Ordner des Notebooks an: `bremsen.csv` (Semikolon, Dezimalkomma), `waschmaschine.csv` (Komma, Dezimalpunkt) und `bremsen-kaputt.csv` (wie die erste, mit fünf kaputten Zeilen). Sie sehen aus wie echte phyphox-Exporte. **Deine eigene Messung** vom Hausauftrag in V08 legst du einfach in denselben Ordner (in Codespaces: in den Dateibaum links ziehen) und trägst ihren Namen ein.

**Regeln:** Nur Python aus V01 bis V09. Kein `sorted`, keine List Comprehension, kein pandas. Schreibe vor dem Ausführen deine **Vorhersage** als Kommentar. KI nur als Erklärer, nicht als Generator.

In [ ]:
# Werkzeug-Zelle: einmal ausführen, nicht verändern.
# Sie legt die Beispieldateien an und definiert die Selbsttests.
import math
import os

def _zahl(x, komma):
    text = "%.6E" % x
    return text.replace(".", ",") if komma else text

def beispieldateien_anlegen():
    kopf = ["Time (s)", "Linear Acceleration x (m/s^2)", "Linear Acceleration y (m/s^2)", "Linear Acceleration z (m/s^2)", "Absolute acceleration (m/s^2)"]
    # Bremsen: 400 Zeilen im Abstand 0.02 s, Bremsung zwischen 4 s und 5.5 s
    zeilen = []
    ay = []
    for i in range(400):
        t = i * 0.02
        rauschen = 0.08 * math.sin(i * 1.7) + 0.05 * math.cos(i * 0.37)
        bremse = 0.0
        if 4.0 <= t <= 5.5:
            bremse = -3.8 * math.sin(math.pi * (t - 4.0) / 1.5) ** 2
        y = round(rauschen + bremse, 6)
        ay.append(y)
        zeilen.append([t, 0.05 * math.sin(i * 0.9), y, 0.04 * math.cos(i * 1.3), abs(y) + 0.1])
    ohne_kaputt = []
    with open("bremsen.csv", "w", encoding="utf-8") as f:
        f.write(";".join('"%s"' % k for k in kopf) + "\n")
        for z in zeilen:
            f.write(";".join(_zahl(v, True) for v in z) + "\n")
    # kaputt: fuenf Zeilen veraendern (leeres Feld, Text, zu kurz)
    with open("bremsen.csv", encoding="utf-8") as f:
        alle = f.read().splitlines()
    for nr, art in [(39, "leer"), (122, "text"), (253, "kurz"), (254, "leer"), (332, "text")]:
        felder = alle[nr - 1].split(";")
        if art == "leer":
            felder[2] = ""
        elif art == "text":
            felder[2] = "Fehler"
        else:
            felder = felder[:2]
        alle[nr - 1] = ";".join(felder)
    with open("bremsen-kaputt.csv", "w", encoding="utf-8") as f:
        f.write("\n".join(alle) + "\n")
    # Waschmaschine: 600 Zeilen im Abstand 0.01 s, Schwingung 12 Hz, Komma und Punkt
    with open("waschmaschine.csv", "w", encoding="utf-8") as f:
        f.write(",".join('"%s"' % k for k in kopf) + "\n")
        for i in range(600):
            t = i * 0.01
            y = 0.9 * math.sin(2 * math.pi * 12 * t) + 0.2 * math.sin(2 * math.pi * 3 * t)
            f.write(",".join(_zahl(v, False) for v in [t, 0.1 * math.sin(i), y, 0.05 * math.cos(i), abs(y)]) + "\n")
    return ay

_ay = beispieldateien_anlegen()
SOLL = {"bremsen_min": round(min(_ay), 2), "bremsen_n": 400}

def pruefe(ist, soll, text):
    if ist == soll:
        print("OK: " + text)
    else:
        print("FEHLER: " + text + " (erwartet " + str(soll) + ", bei dir steht " + str(ist) + ")")

def pruefe_nah(ist, soll, text, toleranz=1e-9):
    # Für Kommazahlen: "gleich" heißt hier, die Abweichung ist winzig (Rundungsfehler, V03).
    if abs(ist - soll) <= toleranz:
        print("OK: " + text)
    else:
        print("FEHLER: " + text + " (erwartet " + str(soll) + ", bei dir steht " + str(ist) + ")")

print("Beispieldateien angelegt:", ", ".join(sorted(f for f in os.listdir(".") if f.endswith(".csv"))))

## Stufe 1 – Aufwärmen

### 1.1 Vorhersagen: strip und split
`strip()` schneidet Leerzeichen und den Zeilenumbruch `\n` an den Enden ab. `split(";")` zerschneidet den Text am Semikolon und liefert eine **Liste aus Texten**. Schreib auf, was jede Zeile ausgibt, und achte darauf, was ein Text und was eine Zahl ist.

In [ ]:
# Meine Vorhersage: ...
zeile = "  4,94;-0,13;-3,78 \n"
print(repr(zeile.strip()))
felder = zeile.strip().split(";")
print(felder)
print(len(felder))
print(felder[2])
print(felder[2] * 2)

### 1.2 Vorhersagen: replace und float
`replace(",", ".")` liefert einen **neuen** Text (der alte bleibt unverändert, Texte lassen sich nicht ändern). Erst danach kennt `float` die Zahl. Welche Zeile bricht ab, und mit welcher Fehlerart? Kommentiere sie dann aus.

In [ ]:
# Meine Vorhersage: ...
text = "-3,78"
print(text.replace(",", "."))
print(text)
print(float(text.replace(",", ".")) * 2)
print(float(text))

### 1.3 Eine Datei schreiben und zurücklesen
Der Modus `"w"` schreibt (und löscht eine vorhandene Datei), `write` schreibt genau den Text, den Zeilenumbruch hängst du selbst an. Wie viele Zeilen hat die Datei am Ende? Was passiert, wenn du in `aus.write(...)` das `\n` weglässt?

In [ ]:
# Meine Vorhersage: ...
with open("notizen.txt", "w", encoding="utf-8") as aus:
    for i in range(3):
        aus.write(f"Messung {i + 1};{i * 0.5:.1f}\n")

with open("notizen.txt", encoding="utf-8") as datei:
    zeilen = datei.readlines()
print(len(zeilen))
print(zeilen)

### 1.4 Selbsttest: eine Zeile in Zahlen zerlegen
Die Zeile stammt aus einer phyphox-Datei mit Semikolon und Dezimalkomma. Setze `zeit` und `a` (Beschleunigung in Spalte 2) auf **Kommazahlen**.

In [ ]:
zeile = "0,060000;0,078954;0,062226;0,111196;0,149901\n"
zeit = None
a = None
# Dein Code hier

pruefe_nah(zeit if zeit is not None else -1, 0.06, "zeit ist 0.06")
pruefe_nah(a if a is not None else -1, 0.062226, "a ist 0.062226")

## Stufe 2 – Anwenden

### 2.1 Die ersten Zeilen einer unbekannten Datei ansehen
Bevor du eine Datei auswertest, schaust du sie roh an. Gib die Kopfzeile und die ersten zwei Messzeilen von `bremsen.csv` mit `repr` aus (so siehst du auch das `\n`). Dann dasselbe mit `waschmaschine.csv`: An welchem Zeichen erkennst du das Trennzeichen? Und das Dezimalzeichen?

In [ ]:
# Dein Code hier: Datei öffnen, readlines, die ersten drei Zeilen mit repr ausgeben

### 2.2 Bremsung auswerten (Semikolon, Dezimalkomma)
Lies `bremsen.csv` mit dem `csv`-Modul (`delimiter=";"`, Kopfzeile mit `next` überspringen) und sammle Zeit (Spalte 0) und Beschleunigung (Spalte 2) in zwei Listen. Danach: kleinster Wert (die Verzögerungsspitze) und sein Zeitpunkt.

In [ ]:
import csv
zeiten = []
werte = []
# Dein Code hier: Datei lesen, Listen füllen

spitze = min(werte) if werte else 0
t_spitze = zeiten[werte.index(spitze)] if werte else 0
print(f"Spitze: {spitze:.2f} m/s² bei t = {t_spitze:.2f} s")
pruefe(len(werte), SOLL["bremsen_n"], "400 Messwerte gelesen")
pruefe(round(spitze, 2), SOLL["bremsen_min"], "Spitze stimmt")

### 2.3 Waschmaschine auswerten (Komma, Dezimalpunkt)
Dieselbe Aufgabe mit `waschmaschine.csv`. Was ändert sich im Code? (Tipp: das Trennzeichen ist der Standard, und ein `replace` ist nicht nötig.) Gib die **Amplitude** aus: den größten Betrag der Beschleunigung in Spalte 2 (`abs`).

In [ ]:
import csv
amplitude = 0.0
# Dein Code hier

print(f"Amplitude: {amplitude:.2f} m/s²")
pruefe(round(amplitude, 1) > 0.5, True, "Amplitude ist größer als 0.5")

### 2.4 Kaputte Zeilen überspringen und zählen
`bremsen-kaputt.csv` hat fünf kaputte Zeilen (leeres Feld, das Wort „Fehler“, eine zu kurze Zeile). Lies sie wie in 2.2, aber fange `ValueError` **und** `IndexError` ab, zähle die kaputten Zeilen in `kaputt` und mache mit `continue` weiter. Das Ergebnis: 395 gültige Zeilen, 5 kaputte.

In [ ]:
import csv
werte = []
kaputt = 0
# Dein Code hier

print(f"Gültige Zeilen: {len(werte)}, übersprungen: {kaputt}")
pruefe(len(werte), 395, "395 gültige Zeilen")
pruefe(kaputt, 5, "5 kaputte Zeilen")

### 2.5 Fehler provozieren und lesen
Mach absichtlich je einen Fehler und lies den Traceback **von unten nach oben**: (a) falscher Dateiname, (b) `float` ohne `replace` bei `bremsen.csv`, (c) `delimiter=","` bei `bremsen.csv` und Zugriff auf `zeile[2]`, (d) eine mit `latin-1` geschriebene Datei mit `encoding="utf-8"` lesen. Welche Fehlerart bekommst du jeweils? Notiere sie in `fehlerarten`.

In [ ]:
fehlerarten = {"a": None, "b": None, "c": None, "d": None}   # z. B. "KeyError" als Text eintragen
# Dein Code hier: vier kleine Versuche (jeweils in try/except, damit das Notebook weiterläuft)
# Tipp: except Exception as fehler: print(type(fehler).__name__)   (siehe Deep Dive der Lerneinheit)

pruefe(fehlerarten["a"], "FileNotFoundError", "a: falscher Dateiname")
pruefe(fehlerarten["b"], "ValueError", "b: Dezimalkomma ohne replace")
pruefe(fehlerarten["c"], "IndexError", "c: falsches Trennzeichen")
pruefe(fehlerarten["d"], "UnicodeDecodeError", "d: falsche Kodierung")

### 2.6 Ergebnis für ein deutsches Excel schreiben
Schreibe `ergebnis.csv`: Kopfzeile `Nr;Wert`, dann die Werte aus `liste` mit **zwei Nachkommastellen und Dezimalkomma**. Lies die Datei danach zurück und prüfe sie.

In [ ]:
liste = [0.5, -1.25, 12.5]
# Dein Code hier: schreiben

with open("ergebnis.csv", encoding="utf-8") as datei:
    zeilen = datei.read().splitlines()
pruefe(zeilen, ["Nr;Wert", "1;0,50", "2;-1,25", "3;12,50"], "ergebnis.csv hat die richtigen Zeilen")

## Stufe 3 – Knobeln

### 3.1 Trennzeichen automatisch erkennen
Schreibe eine Funktion `trennzeichen(dateiname)`, die die zweite Zeile der Datei liest und `";"` zurückgibt, wenn dort ein Semikolon steht, sonst `","`. Teste sie mit allen drei Dateien. (Ein Programm, das beide phyphox-Formate liest, braucht diese Funktion.)

In [ ]:
def trennzeichen(dateiname):
    # Dein Code hier
    return None

pruefe(trennzeichen("bremsen.csv"), ";", "bremsen.csv: Semikolon")
pruefe(trennzeichen("bremsen-kaputt.csv"), ";", "bremsen-kaputt.csv: Semikolon")
pruefe(trennzeichen("waschmaschine.csv"), ",", "waschmaschine.csv: Komma")

### 3.2 Eine Funktion, die jede phyphox-Datei liest
Schreibe `lese_spalte(dateiname, spalte)`: Sie erkennt das Trennzeichen (3.1), wandelt auch ein Dezimalkomma um, überspringt kaputte Zeilen und gibt `(werte, kaputt)` zurück: die Liste der gültigen Werte und die Zahl der kaputten Zeilen. Teste mit allen drei Dateien: Dieselbe Funktion, drei Dateien, ein Aufruf.

In [ ]:
def lese_spalte(dateiname, spalte):
    # Dein Code hier
    return [], 0

w1, k1 = lese_spalte("bremsen.csv", 2)
w2, k2 = lese_spalte("bremsen-kaputt.csv", 2)
w3, k3 = lese_spalte("waschmaschine.csv", 2)
pruefe((len(w1), k1), (400, 0), "bremsen.csv: 400 gültig, 0 kaputt")
pruefe((len(w2), k2), (395, 5), "bremsen-kaputt.csv: 395 gültig, 5 kaputt")
pruefe((len(w3), k3), (600, 0), "waschmaschine.csv: 600 gültig, 0 kaputt")

### 3.3 Frequenz der Waschmaschine aus den Nulldurchgängen
Die Beschleunigung der Waschmaschine pendelt um null. Zähle, wie oft das Vorzeichen wechselt (aufeinanderfolgende Werte mit Produkt kleiner als 0), und berechne daraus die Frequenz: `f = Nulldurchgänge / 2 / Dauer`. Die Dauer ist die letzte Zeit in der Datei. Danach die Drehzahl in U/min (`f * 60`). Beim Beispiel liegt die Frequenz bei etwa 12 Hz.

In [ ]:
f = 0.0
# Dein Code hier (nutze lese_spalte aus 3.2 und die Zeitspalte 0)

print(f"Frequenz: {f:.1f} Hz, Drehzahl: {f * 60:.0f} U/min")
pruefe(round(f), 12, "Frequenz rund 12 Hz")

### 3.4 Deine eigene Messung
Lege deine phyphox-Datei in den Ordner des Notebooks und trage den Namen ein. Schau sie zuerst roh an (2.1), erkenne Trennzeichen und Dezimalzeichen und werte sie mit `lese_spalte` aus: Zahl der Zeilen, Spitze, Mittelwert, Zahl der kaputten Zeilen. Notiere in einer Markdown-Zelle, was dir auffällt (Wann beginnt die Bremsung? Wie stark war sie?).

In [ ]:
dateiname = "bremsen.csv"     # hier deine Datei eintragen
# Dein Code hier

## Rückblick
- Eine Zeile aus einer Datei ist ein **Text**: `strip`, `split`, `replace(",", ".")`, `float` machen Zahlen daraus.
- `with open(name, encoding="utf-8") as datei:` liest und schließt von selbst; `"w"` schreibt und löscht den alten Inhalt.
- `csv.reader(datei, delimiter=...)` zerlegt Zeilen; `next(leser)` holt die Kopfzeile ab.
- Fehler von **unten nach oben** lesen. `try`/`except Fehlerart:` fängt genau diese Art; kaputte Zeilen zählen, nie verschweigen.
- KI-Code zu Dateien prüfst du auf Trennzeichen, Dezimalzeichen, `encoding` und ein nacktes `except`.